<a href="https://colab.research.google.com/github/AndrijaM06/car-price-prediction/blob/main/02_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Čišćenje podataka: cars.csv

Spisak problema iz EDA analize:
1. Nazivi kolona sadrže zagrade (`mileage(kilometers)`, `volume(cm3)`)
2. 87 potpuno identičnih (duplikat) redova
3. Nedostajuće vrednosti u `volume(cm3)`, `drive_unit`, `segment`
4. Nerealna kilometraža (preko 1.000.000 km)
5. Ekstremno stare godine proizvodnje (pre 1970)
6. Ekstremne cene i zapremina motora (outlieri)


## Učitavanje sirovih podataka

In [16]:
import pandas as pd

url = "https://raw.githubusercontent.com/AndrijaM06/car-price-prediction/main/data/cars.csv"
df = pd.read_csv(url)
df.shape

(56244, 12)

In [17]:
df.head()

,make,model,priceUSD,year,condition,mileage(kilometers),fuel_type,volume(cm3),color,transmission,drive_unit,segment
0,mazda,2,5500,2008,with mileage,162000.0,petrol,1500.0,burgundy,mechanics,front-wheel drive,B
1,mazda,2,5350,2009,with mileage,120000.0,petrol,1300.0,black,mechanics,front-wheel drive,B
2,mazda,2,7000,2009,with mileage,61000.0,petrol,1500.0,silver,auto,front-wheel drive,B
3,mazda,2,3300,2003,with mileage,265000.0,diesel,1400.0,white,mechanics,front-wheel drive,B
4,mazda,2,5200,2008,with mileage,97183.0,diesel,1400.0,gray,mechanics,front-wheel drive,B


## 1. Standardizacija naziva kolona

Kolone `mileage(kilometers)` i `volume(cm3)` sadrže zagrade, što nije
praktično za rad u Pythonu. Pretvaramo sve nazive u snake_case i
preimenujemo u čitljivije oblike: `mileage_km`, `price_usd`.

In [18]:
import re

def standardize_column_names(df):
    df = df.copy()
    new_columns = []
    for col in df.columns:
        clean_col = col.strip().lower()
        clean_col = clean_col.replace("(", "_").replace(")", "")
        clean_col = clean_col.replace("-", "_").replace("/", "_")
        clean_col = re.sub(r"\s+", "_", clean_col)
        clean_col = re.sub(r"[^a-z0-9_]", "", clean_col)
        clean_col = re.sub(r"_+", "_", clean_col)
        clean_col = clean_col.strip("_")
        new_columns.append(clean_col)
    df.columns = new_columns
    df = df.rename(columns={"mileage_kilometers": "mileage_km", "priceusd": "price_usd"})
    return df

df = standardize_column_names(df)
df.columns.tolist()

['make',
 'model',
 'price_usd',
 'year',
 'condition',
 'mileage_km',
 'fuel_type',
 'volume_cm3',
 'color',
 'transmission',
 'drive_unit',
 'segment']

**Zaključak:** Kolone su sada u doslednom `snake_case` obliku, bez zagrada, što ih čini mnogo lakšim za dalju upotrebu u kodu (npr. `df.mileage_km` umesto `df["mileage(kilometers)"]`).

## 2. Uklanjanje viška razmaka i standardizacija kategorijskih vrednosti

Proveravamo da li tekstualne kolone imaju razmake na početku/kraju i
pretvaramo kategorijske vrednosti u mala slova radi doslednosti.

In [19]:
text_columns = df.select_dtypes(include=["object", "string"]).columns
for col in text_columns:
    df[col] = df[col].astype("string").str.strip()

categorical_columns = ["make", "model", "condition", "fuel_type", "color", "transmission", "drive_unit", "segment"]
for col in categorical_columns:
    df[col] = df[col].astype("string").str.strip().str.lower()

df[categorical_columns].head()

,make,model,condition,fuel_type,color,transmission,drive_unit,segment
0,mazda,2,with mileage,petrol,burgundy,mechanics,front-wheel drive,b
1,mazda,2,with mileage,petrol,black,mechanics,front-wheel drive,b
2,mazda,2,with mileage,petrol,silver,auto,front-wheel drive,b
3,mazda,2,with mileage,diesel,white,mechanics,front-wheel drive,b
4,mazda,2,with mileage,diesel,gray,mechanics,front-wheel drive,b


## 3. Konverzija numeričkih kolona (zaštitni korak)

In [20]:
numeric_columns = ["price_usd", "year", "mileage_km", "volume_cm3"]
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[numeric_columns].dtypes

,0
price_usd,int64
year,int64
mileage_km,float64
volume_cm3,float64


## 4. Provera i uklanjanje duplikata

In [21]:
duplicate_count = df.duplicated().sum()
print(f"Broj duplikata: {duplicate_count}")

df = df.drop_duplicates()
df.shape

Broj duplikata: 87


(56157, 12)

## 5. Uklanjanje redova bez ciljne vrednosti i sa nevalidnom cenom

Red bez `price_usd` ne može biti trening primer. Takođe uklanjamo redove
gde je cena nula ili negativna, jer takva cena nema smisla za polovni
automobil koji se prodaje.

In [22]:
print("Nedostaje price_usd:", df["price_usd"].isna().sum())
print("Cena <= 0:", (df["price_usd"] <= 0).sum())

df = df.dropna(subset=["price_usd"])
df = df[df["price_usd"] > 0]
df.shape

Nedostaje price_usd: 0
Cena <= 0: 0


(56157, 12)

## 6. Uklanjanje redova sa nerealnom kilometražom

Tokom EDA smo videli da postoje redovi sa kilometražom preko 1.000.000 km
(do skoro 10 miliona), što je fizički nemoguće za putnički automobil.

In [23]:
print("Redova sa kilometražom > 1,000,000 km:", (df["mileage_km"] > 1_000_000).sum())

MAX_REALISTIC_MILEAGE_KM = 1_000_000
df = df[df["mileage_km"].isna() | (df["mileage_km"] <= MAX_REALISTIC_MILEAGE_KM)]
df.shape

Redova sa kilometražom > 1,000,000 km: 363


(55794, 12)

## 7. Uklanjanje redova sa nerealnom godinom proizvodnje

Zadržavamo automobile proizvedene između 1970. i 2020. godine. Stariji
automobili (oldtimeri) imaju potpuno drugačiji obrazac formiranja cene
(kolekcionarska vrednost), što bi zbunilo model.

In [24]:
MIN_REALISTIC_YEAR = 1970
MAX_REALISTIC_YEAR = 2020

print("Redova van opsega godina:", (~df["year"].between(MIN_REALISTIC_YEAR, MAX_REALISTIC_YEAR)).sum())

df = df[df["year"].between(MIN_REALISTIC_YEAR, MAX_REALISTIC_YEAR)]
df.shape

Redova van opsega godina: 122


(55672, 12)

## 8. Uklanjanje redova sa nerealnom zapreminom motora

Zapremina van opsega 300–8000 cm³ nije tipična za putnički automobil.
Nedostajuće vrednosti ovde ne uklanjamo — te ćemo popuniti kasnije, u fazi
pretprocesiranja.

In [25]:
MIN_REALISTIC_VOLUME_CM3 = 300
MAX_REALISTIC_VOLUME_CM3 = 8000

invalid_volume_mask = ~df["volume_cm3"].isna() & ~df["volume_cm3"].between(MIN_REALISTIC_VOLUME_CM3, MAX_REALISTIC_VOLUME_CM3)
print("Redova sa nerealnom zapreminom:", invalid_volume_mask.sum())

df = df[df["volume_cm3"].isna() | df["volume_cm3"].between(MIN_REALISTIC_VOLUME_CM3, MAX_REALISTIC_VOLUME_CM3)]
df = df.reset_index(drop=True)
df.shape

Redova sa nerealnom zapreminom: 87


(55585, 12)

## Provera rezultata čišćenja

Uporedimo dimenzije sirovog i očišćenog skupa podataka, i proverimo da li
i dalje ima nedostajućih vrednosti (to je očekivano za `volume_cm3`,
`drive_unit` i `segment` — te rešavamo tek u pretprocesiranju).

In [26]:
print("Očišćen skup podataka - shape:", df.shape)
df.isna().sum()

Očišćen skup podataka - shape: (55585, 12)


,0
make,0
model,0
price_usd,0
year,0
condition,0
mileage_km,0
fuel_type,0
volume_cm3,47
color,0
transmission,0


In [27]:
df.describe()

,price_usd,year,mileage_km,volume_cm3
count,55585.000000,55585.000000,55585.000000,55538.000000
mean,7463.037726,2003.617325,225875.565999,2081.190356
std,8340.410680,7.863027,126575.580748,721.615475
min,95.000000,1970.000000,0.000000,500.000000
25%,2400.000000,1998.000000,137000.000000,1600.000000
50%,5400.000000,2004.000000,227000.000000,1995.000000
75%,9900.000000,2010.000000,306170.000000,2300.000000
max,235235.000000,2019.000000,1000000.000000,8000.000000


## Zaključak

Od originalnih **56.244** redova, nakon čišćenja ostalo je **55.585**
redova. Uklonili smo:
- 87 duplikata
- redove bez validne cene
- redove sa nerealnom kilometražom (preko 1.000.000 km)
- redove van realnog opsega godina proizvodnje (1970–2020)
- redove sa nerealnom zapreminom motora

Nedostajuće vrednosti u `volume_cm3`, `drive_unit` i `segment` namerno nismo
uklanjali niti popunjavali ovde — to je odgovornost faze pretprocesiranja,
gde koristimo `SimpleImputer`.

Očišćen skup podataka je sačuvan u `data/cars_cleaned.csv` (kroz skript) i
spreman je za sledeći korak: **inženjering karakteristika**.